# 🧠 Scouting Report Agent: RAG Pipeline with Databricks + LangChain

This notebook builds and deploys a Retrieval-Augmented Generation (RAG) agent using:

- **LangChain** for prompt handling and agent logic
- **Databricks Vector Search** to retrieve relevant scouting reports
- **MLflow** to package and deploy the agent
- **Databricks Model Serving** and **Review App** for interactive feedback

---

## 👇 What this notebook covers:

1. 🛠 Install required dependencies  
2. ⚙️ Configure your catalog, schema, model, and index  
3. 🔍 Set up prompt + retrieval logic with player-aware context  
4. 🤖 Define an agentic chain that tracks topics and handles comparisons  
5. 📦 Log and deploy the agent using MLflow  
6. ✅ Test queries and launch the Review App


# 📦 Install LangChain + Databricks SDKs

In [0]:
# Core Databricks + LangChain + MLflow 3
%pip install -U --force-reinstall --quiet\
  mlflow==3.5.1 \
  databricks-agents==1.8.2 \
  databricks-vectorsearch==0.56 \
  databricks-langchain==0.5.1 \
  langchain==0.3.25 \
  langchain-community==0.3.25 \
  langchain-core==0.3.65 \
  cloudpickle==3.1.1 \
  pydantic==2.11.7

# PyData stack versions that play nicely with Python 3.12 on DBR 17.x
%pip install -U --force-reinstall \
  numpy==2.3.4 \
  pandas==2.3.3 \
  pyarrow==19.0.1

dbutils.library.restartPython()


# 🤖 Import Core Libraries

In [0]:
import re
import mlflow
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate

# ⚙️ Set Up Configuration and Agentic Chain

In [0]:
# Define a custom LLM flow that tracks history, detects player changes, and routes retrieval intelligently.”

# Autolog metrics and traces
mlflow.langchain.autolog()

def loader_fn(_context=None):
    """
    Build and return the scouting reports RAG agent as a RunnableLambda
    """

    # --- Configuration ---
    CATALOG = "mlb_tech_summit"
    SCHEMA = "rag_demo"
    INDEX_NAME = "scouting_reports_index"
    LLM_MODEL = "databricks-gpt-5-nano"

    # --- LLM endpoint (Databricks chat model) ---
    # If you get a TypeError on `endpoint=`, switch to `serving_endpoint=LLM_MODEL`.
    llm = ChatDatabricks(endpoint=LLM_MODEL)

    # --- Vector search index (Databricks Vector Search) ---
    vs = DatabricksVectorSearch(
        index_name=f"{CATALOG}.{SCHEMA}.{INDEX_NAME}",
        columns=[
            "primary_key",
            "combined_text",
            "playerName",
            "Season",
            "ID",
            "minorMasterId",
        ],
    )

    # Turn index into a retriever
    retriever = vs.as_retriever(search_kwargs={"k": 1})

    # --- Prompt Template ---
    prompt_template = PromptTemplate(
        input_variables=["context", "question", "chat_history"],
        template="""
You are a baseball scouting assistant. Use the following scouting reports to answer the question.
If the user asks for a comparison, include strengths and weaknesses for each player.

Context:
{context}

Chat History:
{chat_history}

Question: {question}

Answer in full sentences, using only the context provided.
""".strip(),
    )

    # --- Player Name Extractor ---
    # Very lightweight NER-ish heuristic: consecutive capitalized words.
    def extract_all_players(text: str):
        if not text:
            return []
        return re.findall(r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b", text)

    # --- Main chain logic ---
    def run_chain(inputs):
        # 1. Normalize input format
        # Supported:
        #   - list[HumanMessage/AIMessage/...]
        #   - {"query": "...", "messages": [...]}
        #   - "free text string"
        # --- Normalize inputs into (query, messages) ---
        if isinstance(inputs, list) and all(hasattr(m, "content") for m in inputs):
            messages = [
                {
                    "role": "user" if isinstance(m, HumanMessage) else "assistant",
                    "content": m.content,
                }
                for m in inputs[:-1]
            ]
            query = inputs[-1].content

        elif isinstance(inputs, dict):
            # Unwrap {"inputs": {...}} if present
            data = inputs.get("inputs", inputs)

            # Serving endpoint's enforced schema
            required = {"request_id", "source", "text_assessments", "retrieval_assessments"}
            if required.issubset(data.keys()):
                query = (data.get("text_assessments") or "").strip()
                ra = (data.get("retrieval_assessments") or "").strip()
                messages = [{"role": "user", "content": ra}] if ra else []
            else:
                # Generic chat-style dict
                query = (data.get("query") or data.get("question") or "").strip()
                messages = data.get("messages", []) or []

        elif isinstance(inputs, str):
            query = inputs
            messages = []

        else:
            raise ValueError(
                "Unsupported input format. Expected list[BaseMessage], dict, or str."
            )

        # 2. Extract player names
        player_names = extract_all_players(query)

        # 3. Retrieval logic with player switch detection
        prev_user_msgs = [m["content"] for m in messages if m.get("role") == "user"]
        first_msg = prev_user_msgs[0] if prev_user_msgs else None
        initial_players = extract_all_players(first_msg) if first_msg else []
        current_players = extract_all_players(query)

        # Case A: user asked about >1 player in the new question
        if len(current_players) > 1:
            docs = []
            for name in current_players:
                docs.extend(retriever.get_relevant_documents(name))

        # Case B: one named player now
        elif len(current_players) == 1:
            current = current_players[0]
            if initial_players and current != initial_players[0]:
                # user switched players: reset convo context for retrieval
                docs = retriever.get_relevant_documents(current)
                messages = []  # wipe chat history fed to prompt
            else:
                # still same player: retrieve using first + current
                if first_msg:
                    retrieval_query = "\n".join([first_msg, query])
                else:
                    retrieval_query = query
                docs = retriever.get_relevant_documents(retrieval_query)

        # Case C: no explicit player in this turn
        else:
            # use running conversation for retrieval context
            if prev_user_msgs:
                retrieval_query = "\n".join(prev_user_msgs + [query])
            else:
                retrieval_query = query
            docs = retriever.get_relevant_documents(retrieval_query)

        # 4. Format retrieved docs into context block
        def safe_get(season_val):
            # normalize Season just so we don't blow up int(None)
            try:
                if season_val is None:
                    return "N/A"
                return str(int(season_val))
            except Exception:
                return str(season_val)

        context = "\n\n".join(
            f"{doc.metadata.get('playerName', 'Unknown')} report:\n{doc.page_content}"
            for doc in docs
        )

        # 5. Build chat history string for prompt
        chat_history = "\n".join(f"{m['role']}: {m['content']}" for m in messages)

        # 6. Render prompt and call the LLM
        prompt = prompt_template.format(
            question=query,
            chat_history=chat_history,
            context=context,
        )

        llm_response = llm.invoke(prompt)
        answer = (
            llm_response.content
            if hasattr(llm_response, "content")
            else llm_response
        )

        # 7. Attach source metadata for traceability
        sources_md = "\n\n".join(
            (
                f"**Player:** `{doc.metadata.get('playerName', 'Unknown')}`\n"
                f"**Season:** `{safe_get(doc.metadata.get('Season'))}`\n"
                f"**Fangraphs ID:** `{doc.metadata.get('minorMasterId', 'Unknown')}`\n"
                f"**Report ID:** `{doc.metadata.get('primary_key', 'N/A')}`"
            )
            for doc in docs
        )

        # 8. Final markdown response
        return (
            f"### 🧠 Answer\n{answer}\n\n"
            f"---\n"
            f"### 📚 Sources\n{sources_md}"
        )

    # Return a Runnable that MLflow.langchain.log_model can serialize
    return RunnableLambda(run_chain)


# 🧠 Finalize Environment
# Log Run to MLflow Experiment

In [0]:
# Check Core Library Versions
!pip freeze | grep -E 'mlflow|numpy|pandas|pyarrow|langchain|databricks'

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# --- Configuration ---
CATALOG = "mlb_tech_summit"
SCHEMA = "rag_demo"
INDEX_NAME = "scouting_reports_index"
LLM_MODEL = "databricks-gpt-5-nano"

# --- Example input ---
example_input = {
    "query": "Tell me about Ethan Holliday.",
    "messages": []
}

# --- Log the model into MLFlow ---
with mlflow.start_run():
    logged_chain_info = mlflow.langchain.log_model(
        lc_model=loader_fn(),       
        name="abooth_scouting_reports_agent",
        input_example=example_input,
        resources=[
        DatabricksVectorSearchIndex(index_name=f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"),
        DatabricksServingEndpoint(endpoint_name=LLM_MODEL)
        ],
        pip_requirements=[                        # Explicit pip dependencies
            "databricks-agents==1.8.2",
            "databricks-ai-bridge==0.9.0",
            "databricks-automl-runtime==0.2.21",
            "databricks-connect==17.0.10",
            "databricks-feature-engineering==0.12.1",
            "databricks-langchain==0.5.1",
            "databricks-sdk==0.73.0",
            "databricks-vectorsearch==0.56",
            "langchain==0.3.25",
            "langchain-community==0.3.25",
            "langchain-core==0.3.65",
            "langchain-openai==0.3.24",
            "langchain-text-splitters==0.3.8",
            "mlflow==3.5.1",
            "mlflow-skinny==3.5.1",
            "mlflow-tracing==3.5.1",
            "numpy==2.3.4",
            "pandas==2.3.3",
            "pyarrow==19.0.1",
            "unitycatalog-langchain==0.2.0"
        ]
    )

# 🧪 Run Test Queries

In [0]:
# Agent Testing
chain = loader_fn()

# Test with Agent Framework-style input
agent_input = [
    HumanMessage(content="Tell me about Jace LaViolette."),
    AIMessage(content="He’s a left-handed outfielder with power."),
    HumanMessage(content="What are his weaknesses?")
]

print(chain.invoke(agent_input))

print("\n\n")

# Test with direct invocation
example_input = {
    "query": "What are his weaknesses?",
    "messages": [
        {"role": "user", "content": "Tell me about Jace LaViolette."},
        {"role": "assistant", "content": "He’s a left-handed outfielder with power."}
    ]
}

print(chain.invoke(example_input))


In [0]:
# Test with Player Comparisons
agent_input_raw = [HumanMessage(content="Compare Ethan Holliday to Billy Carlson")]
print(chain.invoke(agent_input_raw)) 

In [0]:
# Build the chain
chain = loader_fn()

def show(label, result, max_chars=800):
    print(f"\n=== {label} ===")
    text = result if isinstance(result, str) else str(result)
    print(text[:max_chars] + ("..." if len(text) > max_chars else ""))

# 3) Enforced serving-schema dict
serving_payload = {
    "inputs": {
        "request_id": "1",
        "source": "streamlit",
        "text_assessments": "Tell me about Ethan Holliday",
        "retrieval_assessments": ""
    }
}
res3 = chain.invoke(serving_payload)
show("Serving schema {'request_id','source','text_assessments','retrieval_assessments'}", res3)

# 🚀 Register and Deploy to Databricks Model Serving

In [0]:
from databricks import agents
import mlflow

# Configure MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Config
MODEL_NAME = "scouting_reports_agent"
MODEL_NAME_FQN = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

# Register to UC
logged_model_uri = logged_chain_info.model_uri # update as needed
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_model_uri,
    name=MODEL_NAME_FQN
)


In [0]:
# Deploy the agent to Model Serving
deployment_info = agents.deploy(
    model_name=MODEL_NAME_FQN,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True,
    task="retrieval_qa"
)

# 👀 Create and Configure Access to the Review App

In [0]:
# Add Reviewer Instructions
instructions = """
You are reviewing a baseball scouting assistant. Test it by asking about prospects and their strengths or weaknesses.

- Use full player names.
- Try varying question styles (e.g. 'Is he a good hitter?' vs 'What are his strengths?').
- Use the thumbs up/down to rate documents, and suggest better answers if needed.
"""

agents.set_review_instructions(MODEL_NAME_FQN, instructions)

In [0]:
user_list = ["alexander.booth@databricks.com"]

# Set the permissions.
agents.set_permissions(model_name=MODEL_NAME_FQN, users=user_list, permission_level=agents.PermissionLevel.CAN_QUERY)

In [0]:
from databricks.agents import review_app

# Get the review app
my_app = review_app.get_review_app()

# Print the URL of the review app
print(my_app.url)

## ✅ Review Your Agent in the Review App

Your RAG agent is now deployed and ready for interactive testing!

In the review app, you can:
- Ask questions about players
- Try vague or follow-up questions like “What are his weaknesses?”
- Give feedback using 👍 / 👎
- Suggest better answers or documents

Once you're satisfied, we'll promote the agent to production via Databricks Apps!
